# Solving RLWE with Grover's algorithm

In [1]:
import operator

import numpy as np
import jax.numpy as jnp
from qrisp import *

from qrisp import boolean_simulation, measure, QuantumArray, QuantumModulus, jaspify
from qrisp.alg_primitives.ntt import qntt, qntt_inv, multiply_qntts, multiply_ntts
from qrisp.alg_primitives.ntt import compute_ntt, compute_inv_ntt, multiply_ntts

## The Number Theroretic Transform (NTT)

In [ ]:
# TODO

In [2]:
# Define parameters
n, q, root = 4, 13, 5

In [3]:
@jaspify
def main():

    n, q, root = 4, 13, 5
    a = jnp.array([3, 1, 4, 9])

    qa = QuantumArray(QuantumModulus(q), shape=(n,))
    qa[:] = a

    qntt(qa, root)

    return measure(qa)

print(main())

[10.  7.  9.  8.]                                                               


In [4]:
@jaspify
def main():

    n, q, root = 4, 13, 5
    a = jnp.array([3, 1, 4, 9])
    b = np.array([1, 2, 3, 4])

    qa = QuantumArray(QuantumModulus(q), shape=(n,))
    qa[:] = a

    res = multiply_qntts(qa, b, root)

    return measure(res)

print(main())

[0. 7. 1. 4.]                                                                   


## The Grover attack on RLWE

### The Grover oracle

In [6]:
def condition(v, b_e):
    """Check if the elements of v are within the bounds defined by b_e."""
    n = v.shape[0]
    flag_l = QuantumArray(QuantumBool(), shape=(n,))
    flag_r = QuantumArray(QuantumBool(), shape=(n,))
    flag_lr = QuantumArray(QuantumBool(), shape=(n,))
    flag_all = QuantumBool()
    small_inj_l = flag_l << (lambda v: v <= b_e)
    small_inj_r = flag_r << (lambda v: v >= q - b_e)
    or_inj = flag_lr << (lambda v, w: v | w)
    all_inj = flag_all << (lambda v: v.all())

    with conjugate(small_inj_l)(v):
        with conjugate(small_inj_r)(v):
            with conjugate(or_inj)(flag_l, flag_r):
                with conjugate(all_inj)(flag_lr):
                    z(flag_all)

    flag_l.delete()
    flag_r.delete()
    flag_lr.delete()
    flag_all.delete()


def inv_qntt(x, root):
    with invert():
        qntt(x, root)


def create_lwe_oracle(a_hat, t_hat, q, root, b_e):

    n = a_hat.shape[0]

    def oracle(s_hat):
        #with conjugate(q_ntt(s_hat, n, q, root)): # s -> s_hat
        r_hat = QuantumArray(QuantumModulus(q), shape=(n,))
        inj_multiply_qntts = r_hat << (lambda s_hat, a_hat, root: multiply_qntts(s_hat, a_hat, root))

        with conjugate(inj_multiply_qntts)(s_hat, a_hat, root):

            with conjugate(operator.isub)(r_hat, t_hat):

                with conjugate(inv_qntt)(r_hat, root): # r_hat -> r

                    condition(r_hat, b_e)

        r_hat.delete()

    return oracle


### The amplitude amplification

### Example: n=4, q=13

In [7]:
def lwe_instance(n, q, root):
    a = np.array(np.random.randint(0, q, size=n))
    s = np.array(np.random.randint(0, 2, size=n)) # small secret
    e = np.array(np.random.randint(0, 2, size=n)) # small error

    a_hat = np.array(compute_ntt(a, n, q, root))   
    s_hat = np.array(compute_ntt(s, n, q, root))
    e_hat = np.array(compute_ntt(e, n, q, root))


    t_hat = multiply_ntts(a_hat, s_hat, n, q, root)
    t_hat += e_hat
    t = compute_inv_ntt(t_hat, n, q, root)

    return a, s, e, t

n = 4
q = 13
root = 5

# Set the seed here for reproducible randomness
np.random.seed(42)

a, s, e, t = lwe_instance(n, q, root)
print(a, s, e, t)

[ 6  3 12 10] [1 0 0 0] [1 0 0 0] [ 7  3 12 10]


In [8]:
a_hat = jnp.array(compute_ntt(a, n, q, root))
t_hat = jnp.array(compute_ntt(t, n, q, root))
s_hat = jnp.array(compute_ntt(s, n, q, root))

print(a_hat, s_hat, t_hat)

[ 1  1 11  5] [1 0 1 0] [ 2  1 12  5]


Check that the oracle marks the solution.

In [9]:
oracle = create_lwe_oracle(a_hat, t_hat, q, root=root, b_e=2)

@terminal_sampling
def main():
    qs_hat = QuantumArray(QuantumModulus(q), shape=(n,))
    qs_hat[:] = s_hat

    qb = QuantumBool()
    h(qb)
    with control(qb):
        oracle(qs_hat)
    h(qb)
    return qb

main()

{True: 1.0}

## Amplitude amplification

In [10]:
oracle_func = create_lwe_oracle(a_hat, t_hat, q, root=root, b_e=2)

def state_func(qa):
    for i in range(n):
        qa[i].encode(s_hat[i], permit_dirtyness=True)

    # Bring first entry into superposition (2 element search space)
    ry(3*np.pi/4, qa[0][0])

@terminal_sampling
def main(i):
    qs_hat = QuantumArray(QuantumModulus(q), shape=(n,))
    state_func(qs_hat)

    amplitude_amplification(qs_hat, state_func, oracle_func, iter=i)

    return qs_hat[0]

main(1)

{1: 0.8535524861083176, 0: 0.14644751389168245}

In [ ]:
@jaspify
def solve_rlwe(state_func, a_hat, t_hat, q, root, b_e, iterations):

    n = a_hat.shape[0]
    oracle_func = create_lwe_oracle(a_hat, t_hat, q, root=root, b_e=b_e)

    # Represents s_hat in NTT domain
    qs_hat = QuantumArray(QuantumModulus(q), shape=(n,))
    state_func(qs_hat)

    #amplitude_amplification(qs_hat, state_func, oracle_func, iter=iterations)
    oracle_func(qs_hat)

    return measure(qs_hat)

solve_rlwe(state_func, a_hat, t_hat, q, root, 2, 1)